# AI-Fiber-NLC: Quickstart Demo

> **AI Nonlinear Compensation for Optical Fiber Systems**
>
> This notebook demonstrates the core concept: using AI to compensate for
> fiber nonlinearities that limit modern coherent optical transmission.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Fiber-NLC/AI-Fiber-NLC/blob/main/notebooks/quickstart_demo.ipynb)

## What you will see

1. **Constellation diagrams** — visualize nonlinear distortion before and after compensation
2. **DSP receiver chain** — EDC, clock recovery, carrier phase recovery
3. **DBP baseline** — classical digital back-propagation
4. **Performance comparison** — Q-factor improvement across methods

## Setup

Install dependencies (only needed on first run):

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q OptiCommPy torch numpy matplotlib numba
    !git clone -q https://github.com/AI-Fiber-NLC/AI-Fiber-NLC.git /content/AI-Fiber-NLC
    sys.path.insert(0, '/content/AI-Fiber-NLC')
else:
    sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import torch
import time

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100
print('Setup complete!')

## Step 1: Generate Simulation Data

We simulate a 16QAM, 800km single-polarization fiber link (MVB-1 scene).
The signal is distorted by chromatic dispersion and Kerr nonlinearity,
with EDFA amplification at each 80km span.

This takes about 15 seconds on Colab.

In [ ]:
from src.benchmark.protocol import MVB1
from src.data.simulator import FiberSimulator

scene = MVB1
power_dbm = 1.0  # +1 dBm launch power

print(f'Scene: {scene.name}')
print(f'  Modulation: {scene.modulation}')
print(f'  Distance: {scene.fiber_length_km * scene.num_spans:.0f} km')
print(f'  Power: {power_dbm:+.1f} dBm')
print(f'  Generating data...')

sim = FiberSimulator(scene)
t0 = time.time()
rx, tx = sim.propagate(power_dbm, progress=False)
gen_time = time.time() - t0
print(f'  Generated {len(rx):,} samples in {gen_time:.1f}s')
print(f'  RX power: {np.mean(np.abs(rx)**2):.2e} W')
print(f'  TX power: {np.mean(np.abs(tx)**2):.2e} W')

## Step 2: Visualize Constellation Diagrams

The constellation diagram shows how the transmitted 16QAM symbols are
distorted after 800km of fiber propagation.

In [ ]:
def plot_constellation(sig1, sig2, title='Constellation Diagram', n=3000):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(sig1[:n].real, sig1[:n].imag, s=0.5, alpha=0.6, color='blue')
    axes[0].set_title('Transmitted (Ideal 16QAM)')
    axes[0].set_xlabel('I')
    axes[0].set_ylabel('Q')
    axes[0].axis('equal')
    axes[0].grid(True, alpha=0.3)

    axes[1].scatter(sig2[:n].real, sig2[:n].imag, s=0.5, alpha=0.6, color='red')
    axes[1].set_title('Received (After 800km Fiber)')
    axes[1].set_xlabel('I')
    axes[1].set_ylabel('Q')
    axes[1].axis('equal')
    axes[1].grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_constellation(tx, rx, 'Fiber Nonlinearity: TX vs RX')

## Step 3: DSP Receiver Chain

Apply the full DSP receiver chain:
1. **EDC** (Electronic Dispersion Compensation) — removes chromatic dispersion
2. **Clock Recovery** (Gardner algorithm) — finds optimal sampling instant
3. **Carrier Phase Recovery** (Viterbi-Viterbi) — removes phase rotation

In [ ]:
from src.dsp.receiver import process_receiver

fs = 32e9 * 2  # 2 SPS
m_qam = 16

print('Applying DSP receiver chain...')
result = process_receiver(rx, scene, fs, m_qam=m_qam)
print(f'Baseline Q (EDC+CPR): {result.q_factor_db:+.2f} dB')
print(f'EVM: {result.evm:.4f}')

plot_constellation(result.decided_symbols, result.compensated,
                  'After DSP Chain (EDC + Clock Recovery + CPR)')

## Step 4: DBP Baseline

Digital Back Propagation (DBP) is the classical approach to nonlinear
compensation. It reverses the fiber propagation by numerically solving
the nonlinear Schrodinger equation with inverted coefficients.

In [ ]:
from src.models.baseline_dbp import DBPCompensator

print('Running DBP compensation...')
dbp = DBPCompensator(scene, steps_per_span=10)

t0 = time.time()
dbp_out = dbp.compensate(rx, launch_power_dbm=power_dbm)
dbp_time = time.time() - t0

# DSP chain after DBP (skip EDC since DBP already compensates CD)
dbp_result = process_receiver(dbp_out, scene, fs, m_qam=m_qam, skip_edc=True)
print(f'DBP Q: {dbp_result.q_factor_db:+.2f} dB')
print(f'Improvement: {dbp_result.q_factor_db - result.q_factor_db:+.2f} dB')
print(f'DBP time: {dbp_time:.2f}s')

plot_constellation(dbp_result.decided_symbols, dbp_result.compensated,
                  'After DBP + Clock Recovery + CPR')

## Step 5: Performance Comparison

In [ ]:
print('=' * 60)
print(f'Performance Comparison — {scene.name}, {power_dbm:+.1f} dBm')
print('=' * 60)
print(f'  Baseline (EDC+CPR):  {result.q_factor_db:+.2f} dB')
print(f'  DBP (10 steps/span): {dbp_result.q_factor_db:+.2f} dB')
print(f'  Improvement:         {dbp_result.q_factor_db - result.q_factor_db:+.2f} dB')
print('=' * 60)

methods = ['Baseline\n(EDC+CPR)', 'DBP\n(10 steps/span)']
qs = [result.q_factor_db, dbp_result.q_factor_db]
colors = ['#3498db', '#e74c3c']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, qs, color=colors, width=0.5, edgecolor='white', linewidth=2)
for bar, q in zip(bars, qs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{q:+.2f} dB', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Q-factor (dB)')
ax.set_title(f'NLC Performance Comparison\n{scene.name}, {power_dbm:+.1f} dBm')
ax.axhline(y=0, color='gray', linewidth=0.5, linestyle='--')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nKey finding: At +1 dBm launch power, nonlinear distortion is weak.')
print('EDC already removes chromatic dispersion (linear impairment).')
print('DBP advantage appears at higher powers (>+5 dBm) or longer distances.')
print('This is the problem AI-NLC aims to solve: better nonlinear compensation')
print('at lower computational cost than DBP.')

## What's Next?

This demo shows the foundation. The full project includes:

- **Multiple scenarios**: MVB-1 (16QAM), MVB-2 (DP-16QAM), MVB-3 (64QAM+PCS)
- **AI models**: MLP-NLC, CNN-NLC, Transformer-NLC, KAN-NLC (Phase 2)
- **Benchmark protocol**: standardized scoring with Q-factor + FLOPs
- **Contributor framework**: decentralized training with contribution tracking

**Get involved:** https://github.com/AI-Fiber-NLC/AI-Fiber-NLC